In [1]:
import os 
os.chdir("../")
%pwd

'd:\\Programming\\ML\\End-to-End\\End-to-End-TelcoChurn'

In [32]:
from dataclasses import dataclass 
from pathlib import Path 

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path 
    data_path: Path
    train_path: Path 
    test_path: Path
    transformer_path: Path
    

In [33]:
from src.constants import *
from src.utils import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self,
                config_path = CONFIG_FILE_PATH,
                params_path = PARAMS_FILE_PATH,
                schema_path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)
        self.schema = read_yaml(schema_path)
        
        create_directories([self.config.artifacts_root])
        
        
    def get_data_transformation_config(self)-> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])
        
        return DataTransformationConfig(
            root_dir= Path(config.root_dir),
            data_path=Path(config.data_path),
            train_path=Path(config.train_path),
            test_path=Path(config.test_path),
            transformer_path=Path(config.transformer_path)
        )

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, FunctionTransformer
from src import logging
import joblib

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def get_data_transformer_object(self, df):
        logging.info("Getting data transformer object dynamically")

        # --- Identify column types dynamically ---
        numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
        object_cols = df.select_dtypes(include=['object']).columns.tolist()

        # If TotalCharges is 'object' but should be numeric
        if 'TotalCharges' in object_cols:
            numerical_cols.append('TotalCharges')
            object_cols.remove('TotalCharges')

        # Example of binary vs multi-categorical
        binary_cols = [col for col in object_cols if df[col].nunique() == 2]
        multi_category_cols = [col for col in object_cols if df[col].nunique() > 2]

        logging.info(f"Numerical cols: {numerical_cols}")
        logging.info(f"Binary cols: {binary_cols}")
        logging.info(f"Multi-category cols: {multi_category_cols}")

        # --- Define pipelines ---
        # 1️⃣ Convert TotalCharges to numeric dynamically
        def to_numeric(df):
            df = df.copy()
            if 'TotalCharges' in df.columns:
                df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
            return df

        numeric_transformer = Pipeline(steps=[
            ('to_numeric', FunctionTransformer(to_numeric)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])

        binary_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder())
        ])

        multi_cat_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ])

        # --- Combine all transformers ---
        preprocessor = ColumnTransformer(transformers=[
            ('num', numeric_transformer, numerical_cols),
            ('bin', binary_transformer, binary_cols),
            ('multi', multi_cat_transformer, multi_category_cols)
        ])
        

        
        return preprocessor


In [35]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.get_data_transformer_object()
except Exception as e:
    raise e

[2025-10-15 11:42:04,102] [INFO] [root:read_yaml:16] - reading the content of 'config\config.yaml'
[2025-10-15 11:42:04,104] [INFO] [root:read_yaml:16] - reading the content of 'params.yaml'


[2025-10-15 11:42:04,107] [INFO] [root:read_yaml:16] - reading the content of 'schema.yaml'
[2025-10-15 11:42:04,108] [INFO] [root:create_directories:39] - created directory at: artifacts
[2025-10-15 11:42:04,110] [INFO] [root:create_directories:39] - created directory at: artifacts/data_transformation


TypeError: DataTransformation.get_data_transformer_object() missing 1 required positional argument: 'df'

In [ ]:
['MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaymentMethod',
]

AttributeError: 'list' object has no attribute 'list'